# Hierarchical World Model Demo

This notebook demonstrates the hierarchical world model implementation, including:
- Model architecture (Encoder, Decoder, Actor)
- Training on CIFAR-10 dataset
- Evaluation and visualization

This notebook is compatible with both **Google Colab** and **Jupyter Notebook**.

## 1. Setup and Installation

In [ ]:
import sys
import os

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

# Repository URL for this project (used for Colab setup)
REPO_URL = 'https://github.com/rm-2278/hierarchical-world-models.git'

if IN_COLAB:
    print("Running in Google Colab")
    # Clone the repository if not already present
    if not os.path.exists('hierarchical-world-models'):
        !git clone {REPO_URL}
    %cd hierarchical-world-models
    !pip install -q -r requirements.txt
else:
    print("Running in Jupyter Notebook")
    # Navigate to project root (assuming notebook is in /notebooks)
    os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
    
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Add source directory to path
PROJECT_ROOT = os.getcwd() if os.path.basename(os.getcwd()) != 'notebooks' else os.path.dirname(os.getcwd())
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Source directory: {SRC_DIR}")

In [ ]:
# Import libraries
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch import nn, optim
from torch.utils.data import DataLoader

# Import project modules
from models.model_demo import Agent, SimpleEncoder, SimpleDecoder, SimpleActor
from data.dataset_demo import CIFAR10Dataset

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Model Architecture Overview

The hierarchical world model consists of three main components:

1. **SimpleEncoder**: Encodes 64x64 RGB images into a latent representation
2. **SimpleDecoder**: Reconstructs images from latent representations  
3. **SimpleActor**: Predicts actions from latent representations

The `Agent` class combines all three components for easy use.

In [ ]:
# Visualize model architecture
latent_dim = 128
action_dim = 4

encoder = SimpleEncoder(latent_dim=latent_dim)
decoder = SimpleDecoder(latent_dim=latent_dim)
actor = SimpleActor(latent_dim=latent_dim, action_dim=action_dim)

print("=" * 50)
print("SimpleEncoder Architecture:")
print("=" * 50)
print(encoder)
print(f"\nInput: (batch, 3, 64, 64) -> Output: (batch, {latent_dim})")

print("\n" + "=" * 50)
print("SimpleDecoder Architecture:")
print("=" * 50)
print(decoder)
print(f"\nInput: (batch, {latent_dim}) -> Output: (batch, 3, 64, 64)")

print("\n" + "=" * 50)
print("SimpleActor Architecture:")
print("=" * 50)
print(actor)
print(f"\nInput: (batch, {latent_dim}) -> Output: (batch, {action_dim})")

In [ ]:
# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Encoder parameters: {count_parameters(encoder):,}")
print(f"Decoder parameters: {count_parameters(decoder):,}")
print(f"Actor parameters: {count_parameters(actor):,}")
print(f"Total parameters: {count_parameters(encoder) + count_parameters(decoder) + count_parameters(actor):,}")

## 3. Load and Visualize Dataset

We use CIFAR-10 dataset, resized to 64x64 for the model.

In [ ]:
# Load dataset
print("Loading CIFAR-10 dataset...")
train_dataset = CIFAR10Dataset(train=True, img_size=64)
test_dataset = CIFAR10Dataset(train=False, img_size=64)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
                 'dog', 'frog', 'horse', 'ship', 'truck']

for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img)
    ax.set_title(cifar_classes[label])
    ax.axis('off')

plt.suptitle('CIFAR-10 Sample Images (64x64)', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Training the Model

We train the encoder-decoder as an autoencoder using MSE loss for image reconstruction.

In [ ]:
# Training configuration
config = {
    'latent_dim': 128,
    'action_dim': 4,
    'batch_size': 64,
    'learning_rate': 1e-3,
    'epochs': 5,  # Reduced for demo (use more epochs for better results)
    'seed': 42
}

# Set seed for reproducibility
torch.manual_seed(config['seed'])
np.random.seed(config['seed'])

print("Training Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

In [ ]:
# Initialize agent and training components
agent = Agent(
    device=device,
    latent_dim=config['latent_dim'],
    action_dim=config['action_dim']
)

# Create data loader
# Note: num_workers=0 for Colab compatibility; increase for faster local training
train_loader = DataLoader(
    train_dataset, 
    batch_size=config['batch_size'], 
    shuffle=True,
    num_workers=0 if IN_COLAB else 2
)

# Optimizer and loss
params = list(agent.encoder.parameters()) + list(agent.decoder.parameters())
optimizer = optim.Adam(params, lr=config['learning_rate'])
criterion = nn.MSELoss()

print(f"Agent initialized on {device}")
print(f"Number of batches per epoch: {len(train_loader)}")

In [ ]:
# Training loop
losses = []

print("Starting training...\n")
for epoch in range(config['epochs']):
    agent.encoder.train()
    agent.decoder.train()
    
    epoch_loss = 0.0
    for batch_idx, (imgs, _) in enumerate(train_loader):
        # Prepare images: (B, H, W, C) -> (B, C, H, W)
        imgs_tensor = imgs.float().permute(0, 3, 1, 2).to(device)
        
        # Forward pass
        z = agent.encoder(imgs_tensor)
        recon = agent.decoder(z)
        
        # Compute loss
        loss = criterion(recon, imgs_tensor)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * imgs_tensor.size(0)
        
        # Print progress
        if (batch_idx + 1) % 200 == 0:
            print(f"  Epoch {epoch+1}/{config['epochs']}, Batch {batch_idx+1}/{len(train_loader)}, Loss: {loss.item():.6f}")
    
    avg_loss = epoch_loss / len(train_dataset)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{config['epochs']} - Average Loss: {avg_loss:.6f}")

print("\nTraining complete!")

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(losses) + 1), losses, 'b-o', linewidth=2, markersize=8)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Reconstruction Loss (MSE)', fontsize=12)
plt.title('Training Loss Over Epochs', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Evaluation and Visualization

Let's evaluate the trained model on test images and visualize reconstructions.

In [ ]:
# Evaluate on test set
agent.encoder.eval()
agent.decoder.eval()

test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)

test_loss = 0.0
with torch.no_grad():
    for imgs, _ in test_loader:
        imgs_tensor = imgs.float().permute(0, 3, 1, 2).to(device)
        z = agent.encoder(imgs_tensor)
        recon = agent.decoder(z)
        loss = criterion(recon, imgs_tensor)
        test_loss += loss.item() * imgs_tensor.size(0)

avg_test_loss = test_loss / len(test_dataset)
print(f"Test Set Reconstruction Loss: {avg_test_loss:.6f}")

In [ ]:
# Visualize reconstructions
n_samples = 8
fig, axes = plt.subplots(2, n_samples, figsize=(16, 4))

# Get random test samples
indices = np.random.choice(len(test_dataset), n_samples, replace=False)

for i, idx in enumerate(indices):
    img, label = test_dataset[idx]
    
    # Get reconstruction using agent
    _, recon = agent(img, eval=True)
    
    # Original image
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'{cifar_classes[label]}', fontsize=10)
    axes[0, i].axis('off')
    
    # Reconstructed image
    axes[1, i].imshow(np.clip(recon, 0, 1))
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('Reconstructed', fontsize=12)

plt.suptitle('Original vs Reconstructed Images', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Latent Space Exploration

Let's explore the learned latent space.

In [ ]:
# Encode a batch of images and visualize latent representations
agent.encoder.eval()

# Get embeddings for some samples
n_viz = 500
embeddings = []
labels_list = []

with torch.no_grad():
    for i in range(n_viz):
        img, label = test_dataset[i]
        img_tensor = torch.tensor(img).float().permute(2, 0, 1).unsqueeze(0).to(device)
        z = agent.encoder(img_tensor)
        embeddings.append(z.cpu().numpy().flatten())
        labels_list.append(label)

embeddings = np.array(embeddings)
labels_array = np.array(labels_list)

print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# Simple PCA for visualization (using first 2 principal components)
# Center the data
embeddings_centered = embeddings - embeddings.mean(axis=0)

# Compute covariance and eigenvectors
cov_matrix = np.cov(embeddings_centered.T)
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Sort by eigenvalue (descending)
idx = np.argsort(eigenvalues)[::-1]
eigenvectors = eigenvectors[:, idx]

# Project to 2D
embeddings_2d = embeddings_centered @ eigenvectors[:, :2]

# Plot
plt.figure(figsize=(10, 8))
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                      c=labels_array, cmap='tab10', alpha=0.7, s=30)
plt.colorbar(scatter, ticks=range(10), label='Class')
plt.xlabel('First Principal Component', fontsize=12)
plt.ylabel('Second Principal Component', fontsize=12)
plt.title('Latent Space Visualization (PCA)', fontsize=14)

# Add class legend
for i, cls in enumerate(cifar_classes):
    mask = labels_array == i
    if mask.any():
        center = embeddings_2d[mask].mean(axis=0)

plt.tight_layout()
plt.show()

## 7. Action Prediction Demo

The Actor network predicts actions from the latent representation.

In [ ]:
# Demonstrate action prediction
agent.actor.eval()

print("Action Predictions for Sample Images:")
print("=" * 60)

for i in range(5):
    img, label = test_dataset[i]
    action, _ = agent(img, eval=True)
    
    print(f"\nImage {i+1} ({cifar_classes[label]}):")
    print(f"  Action vector: [{', '.join([f'{a:.4f}' for a in action])}]")
    print(f"  Action mean: {action.mean():.4f}, std: {action.std():.4f}")

## 8. Save and Load Model

Demonstrate how to save and load trained models.

In [ ]:
# Save the trained agent
save_dir = os.path.join(PROJECT_ROOT, 'experiments', 'results', 'demo_notebook')
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, 'agent_demo.pth')
agent.save(save_path)
print(f"Agent saved to: {save_path}")

In [ ]:
# Load the agent and verify
loaded_agent = Agent.load(
    save_path, 
    device=device,
    latent_dim=config['latent_dim'],
    action_dim=config['action_dim']
)

# Test loaded agent
test_img, test_label = test_dataset[0]
original_action, original_recon = agent(test_img, eval=True)
loaded_action, loaded_recon = loaded_agent(test_img, eval=True)

print("Verification - Comparing original and loaded agent outputs:")
print(f"  Action difference: {np.abs(original_action - loaded_action).max():.10f}")
print(f"  Reconstruction difference: {np.abs(original_recon - loaded_recon).max():.10f}")
print("\nModel loaded successfully!" if np.allclose(original_action, loaded_action) else "Warning: Outputs differ!")

## 9. Summary

In this notebook, we demonstrated:

1. **Setup**: Compatible with both Google Colab and Jupyter Notebook
2. **Architecture**: SimpleEncoder, SimpleDecoder, and SimpleActor components
3. **Training**: Autoencoder training on CIFAR-10 dataset
4. **Evaluation**: Test set reconstruction quality
5. **Visualization**: Original vs reconstructed images, latent space
6. **Action Prediction**: Using the Actor network
7. **Model Persistence**: Save and load functionality

### Next Steps

- Train for more epochs for better reconstruction quality
- Experiment with different latent dimensions
- Try different datasets or environments
- Implement hierarchical components for world modeling

In [ ]:
# Clean up (optional)
print("Demo complete!")
print(f"\nResults saved in: {save_dir}")